# Patch Selection Strategy Experiment: DeepDetect
Standalone experiment comparing five frequency extraction strategies.
No changes are made to the codebase - all logic lives in this notebook.
The best strategy will be transferred to the codebase after evaluation.

| ID | Strategy | Description |
|---|---|---|
| v1 | Variance only | Original — select lowest-variance patch |
| v2 | Brightness filtered | Filter dark patches first, then select flattest |
| v3 | Spectral entropy | Select patch with most structured frequency signature |
| v4 | High-pass FFT | No patch — full image with DC and low frequencies masked |
| v5 | Skin tone guided | Prefer patches in skin-tone regions (HSV) |

Each strategy runs `run_freq_only_baseline()` for 20 epochs on DeepDetect.
Results compared by test set accuracy. Threshold: >= 70% pass, 60-70% weak, < 60% fail.


## 1. Setup

In [7]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import torch
import torch.nn.functional as F
import numpy as np
import importlib

print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


Device: cuda
GPU: NVIDIA RTX PRO 4000 Blackwell


## 2. Data

In [8]:
from config import Config
from data.deepdetect import get_deepdetect_loaders

def make_cfg(experiment_name):
    cfg = Config()
    cfg.data.deepdetect_root   = "../data/raw/deep_detect/data"
    cfg.data.dataset           = "deepdetect"
    cfg.data.image_size        = 224
    cfg.data.batch_size        = 96
    cfg.data.num_workers       = 4
    cfg.backbone.img_size      = 224
    cfg.frequency.patch_size   = 56
    cfg.data.jpeg_aug          = True
    cfg.data.blur_aug          = True
    cfg.data.noise_aug         = True
    cfg.data.recompression_aug = True
    cfg.data.mixed_aug         = True
    cfg.data.mixed_aug_prob    = 0.3
    cfg.train.epochs           = 20
    cfg.train.backbone_lr      = cfg.train.lr
    cfg.experiment_name        = experiment_name
    cfg.notes                  = f"patch experiment: {experiment_name}"
    return cfg

cfg_data = make_cfg("patch_exp_v1")
train_loader, val_loader, test_loader = get_deepdetect_loaders(cfg_data)


Train: 76,848  Val: 13,561  Test: 21,776


## 3. Strategy Implementations
All five strategies defined here. None touch the codebase.
v4 (high-pass FFT) requires a different approach since it replaces the entire
FFT pipeline rather than just the patch selector - handled separately below.


In [ ]:
import torch
import torch.nn.functional as F

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

def denorm(t):
    return (t * IMAGENET_STD + IMAGENET_MEAN).clamp(0, 1)

def select_patch_v1(image, patch_size=56):
    C, H, W = image.shape
    patch_size = min(patch_size, H, W)
    if patch_size == H and patch_size == W:
        return image
    gray = image.mean(dim=0, keepdim=True).unsqueeze(0)
    kernel = torch.ones(1, 1, patch_size, patch_size,
                        device=image.device) / (patch_size ** 2)
    local_mean_sq = F.conv2d(gray ** 2, kernel, padding=0)
    local_mean    = F.conv2d(gray,      kernel, padding=0)
    local_var     = (local_mean_sq - local_mean ** 2).squeeze()
    flat_idx = local_var.argmin()
    top  = flat_idx // local_var.shape[1]
    left = flat_idx  % local_var.shape[1]
    return image[:, top:top+patch_size, left:left+patch_size]

def select_patch_v2(image, patch_size=56, min_brightness=0.15):
    C, H, W = image.shape
    patch_size = min(patch_size, H, W)
    if patch_size == H and patch_size == W:
        return image

    gray   = image.mean(dim=0, keepdim=True).unsqueeze(0)  # (1,1,H,W)
    kernel = torch.ones(1, 1, patch_size, patch_size,
                        device=image.device) / (patch_size ** 2)

    local_mean    = F.conv2d(gray,      kernel, padding=0)
    local_mean_sq = F.conv2d(gray ** 2, kernel, padding=0)
    local_var     = (local_mean_sq - local_mean ** 2).squeeze()  # (H', W')
    bright = image.mean(dim=0, keepdim=True).unsqueeze(0)
    local_brightness = F.conv2d(bright, kernel, padding=0).squeeze()  # (H', W')

    bright_mask = (local_brightness >= min_brightness)

    if bright_mask.any():
        # Set dark patches to inf so they are never selected
        local_var_masked = local_var.clone()
        local_var_masked[~bright_mask] = float("inf")
        flat_idx = local_var_masked.argmin()
    else:
        # Fall back to v1 if no bright patch exists
        flat_idx = local_var.argmin()

    top  = flat_idx // local_var.shape[1]
    left = flat_idx  % local_var.shape[1]
    return image[:, top:top+patch_size, left:left+patch_size]

def spectral_entropy(patch):
    gray = patch.mean(0)
    spec = torch.fft.fft2(gray.float())
    spec = torch.fft.fftshift(spec)
    mag  = torch.abs(spec)
    mag  = mag / (mag.sum() + 1e-8)
    return -(mag * torch.log(mag + 1e-8)).sum().item()

def select_patch_v3(image, patch_size=56, stride=16):
    # stride=16 for speed — FFT per patch is expensive
    C, H, W = image.shape
    patch_size = min(patch_size, H, W)
    if patch_size == H and patch_size == W:
        return image
    best_entropy, best_patch = float("inf"), None
    for y in range(0, H - patch_size + 1, stride):
        for x in range(0, W - patch_size + 1, stride):
            patch   = image[:, y:y+patch_size, x:x+patch_size]
            entropy = spectral_entropy(patch)
            if entropy < best_entropy:
                best_entropy, best_patch = entropy, patch
    return best_patch

def rgb_to_hsv(rgb):
    r, g, b = rgb[0], rgb[1], rgb[2]
    maxc = rgb.max(0).values
    minc = rgb.min(0).values
    v    = maxc
    s    = torch.where(maxc != 0, (maxc - minc) / (maxc + 1e-8),
                       torch.zeros_like(maxc))
    rc   = (maxc - r) / (maxc - minc + 1e-8)
    gc   = (maxc - g) / (maxc - minc + 1e-8)
    bc   = (maxc - b) / (maxc - minc + 1e-8)
    h    = torch.where(r == maxc, bc - gc,
           torch.where(g == maxc, 2.0 + rc - bc, 4.0 + gc - rc))
    h    = (h / 6.0) % 1.0
    return h, s, v

def select_patch_v5(image, patch_size=56, min_skin_density=0.3):
    C, H, W = image.shape
    patch_size = min(patch_size, H, W)
    if patch_size == H and patch_size == W:
        return image

    # Denorm to get actual pixel values for HSV
    img_denorm = denorm(image.cpu())
    h, s, v    = rgb_to_hsv(img_denorm)
    
    # Skin mask — (H, W) binary
    mask = ((h >= 0.0) & (h <= 0.1) & 
            (s >= 0.1) & (s <= 0.7) & 
            (v >= 0.2)).float().unsqueeze(0).unsqueeze(0)  # (1,1,H,W)

    # Compute variance map via conv2d
    gray   = image.mean(dim=0, keepdim=True).unsqueeze(0)
    kernel = torch.ones(1, 1, patch_size, patch_size,
                        device=image.device) / (patch_size ** 2)
    local_mean    = F.conv2d(gray,      kernel, padding=0)
    local_mean_sq = F.conv2d(gray ** 2, kernel, padding=0)
    local_var     = (local_mean_sq - local_mean ** 2).squeeze()  # (H', W')

    # Compute skin density map via conv2d on the mask
    skin_kernel = torch.ones(1, 1, patch_size, patch_size) / (patch_size ** 2)
    local_skin  = F.conv2d(mask, skin_kernel, padding=0).squeeze()  # (H', W')

    # Prefer patches with sufficient skin coverage
    skin_present = (local_skin >= min_skin_density)

    if skin_present.any():
        local_var_masked = local_var.clone()
        local_var_masked[~skin_present] = float("inf")
        flat_idx = local_var_masked.argmin()
    else:
        # Fall back to v1 if no skin region found
        flat_idx = local_var.argmin()

    top  = flat_idx // local_var.shape[1]
    left = flat_idx  % local_var.shape[1]
    return image[:, top:top+patch_size, left:left+patch_size]

## 4. v4: High-Pass FFT Implementation
v4 replaces the entire FFT pipeline rather than just patch selection.
We monkey-patch `fft_spectrum_tensor` in `fft_utils` to apply a radial
high-pass mask to the full image spectrum instead of a patch.


In [10]:
def highpass_fft_spectrum(image_tensor, fftshift=True, cutoff_fraction=0.1):
    """
    High-pass FFT on the full image.
    Masks out DC component and low-frequency ring.
    No patch selection needed.

    Args:
        cutoff_fraction: fraction of spectrum radius to zero out.
                         0.1 = mask inner 10% of spectrum (DC + low freq).
    """
    # Cast to float32 — avoids cuFFT half-precision restriction
    spectrum = torch.fft.fft2(image_tensor.float())
    if fftshift:
        spectrum = torch.fft.fftshift(spectrum, dim=(-2, -1))

    # Build radial high-pass mask
    B, C, H, W = image_tensor.shape
    cy, cx     = H // 2, W // 2
    ys = torch.arange(H, device=image_tensor.device).float() - cy
    xs = torch.arange(W, device=image_tensor.device).float() - cx
    radius     = torch.sqrt(ys[:, None] ** 2 + xs[None, :] ** 2)
    cutoff     = cutoff_fraction * min(H, W) / 2
    mask       = (radius > cutoff).float()          # 1 = keep, 0 = mask out
    mask       = mask.unsqueeze(0).unsqueeze(0)     # (1, 1, H, W)

    spectrum   = spectrum * mask

    log_magnitude = torch.log1p(torch.abs(spectrum))

    # Normalise per sample
    mean = log_magnitude.mean(dim=(-2,-1), keepdim=True)
    std  = log_magnitude.std(dim=(-2,-1), keepdim=True)
    return (log_magnitude - mean) / (std + 1e-8)

## 5. Runner

In [14]:
import utils.patch_select as ps_module
import utils.fft_utils    as fft_module
import models.frequency_branch as fb_module

def run_patch_strategy(strategy_fn, strategy_name, cfg):
    """Run freq-only baseline with a custom patch selector."""
    from utils.patch_select import select_flat_patch_batch as original_fn

    def patched_batch(images, patch_size=56):
        return torch.stack([strategy_fn(images[i], patch_size) for i in range(images.shape[0])])

    ps_module.select_flat_patch_batch = patched_batch
  
    importlib.reload(fb_module)

    try:
        print(f"\n{'='*60}\nRunning: {strategy_name}\n{'='*60}")
        from experiments.baseline_freq_only import run_freq_only_baseline
        return run_freq_only_baseline(cfg, train_loader, val_loader, test_loader)
    finally:
        ps_module.select_flat_patch_batch = original_fn
        importlib.reload(fb_module)


def run_highpass_strategy(strategy_name, cfg):
    """Run freq-only baseline with high-pass FFT replacing patch pipeline."""
    original_fft = fft_module.fft_spectrum_tensor

    # Wrap to accept same signature but ignore patch — operates on full image
    def patched_fft(image_tensor, fftshift=True):
        return highpass_fft_spectrum(image_tensor, fftshift=fftshift)

    fft_module.fft_spectrum_tensor = patched_fft
    import models.frequency_branch as fb_module
    importlib.reload(fb_module)

    try:
        print(f"\n{'='*60}\nRunning: {strategy_name}\n{'='*60}")
        from experiments.baseline_freq_only import run_freq_only_baseline
        return run_freq_only_baseline(cfg, train_loader, val_loader, test_loader)
    finally:
        fft_module.fft_spectrum_tensor = original_fft
        importlib.reload(fb_module)

## 6. Run All Strategies


In [6]:
results = {}


In [7]:
cfg_v1 = make_cfg("patch_exp_v1")
results["v1 — variance only"] = run_patch_strategy(
    select_patch_v1, "v1 — variance only", cfg_v1
)



Running: v1 — variance only

Experiment: patch_exp_v1
Frequency-only baseline | Epochs: 20
Train: 76,848  Val: 13,561



Epoch   1/20 | train_loss=0.6905 | val_acc=51.5%
  -> Saved best val_acc=51.5%


Epoch   2/20 | train_loss=0.6868 | val_acc=55.2%
  -> Saved best val_acc=55.2%


Epoch   3/20 | train_loss=0.6848 | val_acc=52.3%


Epoch   4/20 | train_loss=0.6830 | val_acc=54.6%


...

Epoch  16/20 | train_loss=0.6689 | val_acc=58.0%
  -> Saved best val_acc=58.0%


Epoch  17/20 | train_loss=0.6689 | val_acc=57.6%


Epoch  18/20 | train_loss=0.6674 | val_acc=57.9%


Epoch  19/20 | train_loss=0.6667 | val_acc=57.6%


Epoch  20/20 | train_loss=0.6669 | val_acc=57.7%

Frequency-only final results:
  Accuracy: 58.2%
  AUC-ROC:  0.628
  F1:       0.367


Result: FAIL — frequency branch is below 60%.
Results saved → ./results/results.csv  (patch_exp_v1, acc=0.582)


In [8]:
cfg_v2 = make_cfg("patch_exp_v2")
results["v2 — brightness filtered"] = run_patch_strategy(
    select_patch_v2, "v2 — brightness filtered", cfg_v2
)



Running: v2 — brightness filtered

Experiment: patch_exp_v2
Frequency-only baseline | Epochs: 20
Train: 76,848  Val: 13,561



Epoch   1/20 | train_loss=0.6857 | val_acc=48.9%
  -> Saved best val_acc=48.9%


Epoch   2/20 | train_loss=0.6755 | val_acc=49.2%
  -> Saved best val_acc=49.2%


Epoch   3/20 | train_loss=0.6698 | val_acc=57.6%
  -> Saved best val_acc=57.6%


Epoch   4/20 | train_loss=0.6657 | val_acc=59.0%
  -> Saved best val_acc=59.0%


...

Epoch  16/20 | train_loss=0.6378 | val_acc=61.6%


Epoch  17/20 | train_loss=0.6354 | val_acc=61.6%


Epoch  18/20 | train_loss=0.6340 | val_acc=61.2%


Epoch  19/20 | train_loss=0.6321 | val_acc=61.1%


Epoch  20/20 | train_loss=0.6330 | val_acc=61.2%

Frequency-only final results:
  Accuracy: 62.9%
  AUC-ROC:  0.656
  F1:       0.455

Result: WEAK — frequency branch is below the 70% target (60-70%).
Results saved → ./results/results.csv  (patch_exp_v2, acc=0.6293)


In [9]:
cfg_v4 = make_cfg("patch_exp_v4")
results["v4 — high-pass FFT"] = run_highpass_strategy(
    "v4 — high-pass FFT", cfg_v4
)



Running: v4 — high-pass FFT

Experiment: patch_exp_v4
Frequency-only baseline | Epochs: 20
Train: 76,848  Val: 13,561



Epoch   1/20 | train_loss=0.6904 | val_acc=54.0%
  -> Saved best val_acc=54.0%


Epoch   2/20 | train_loss=0.6865 | val_acc=54.0%
  -> Saved best val_acc=54.0%


Epoch   3/20 | train_loss=0.6858 | val_acc=54.0%
  -> Saved best val_acc=54.0%


Epoch   4/20 | train_loss=0.6841 | val_acc=54.8%
  -> Saved best val_acc=54.8%


...

Epoch  16/20 | train_loss=0.6717 | val_acc=56.7%
  -> Saved best val_acc=56.7%


Epoch  17/20 | train_loss=0.6706 | val_acc=56.1%


Epoch  18/20 | train_loss=0.6716 | val_acc=56.6%


Epoch  19/20 | train_loss=0.6709 | val_acc=56.9%
  -> Saved best val_acc=56.9%


Epoch  20/20 | train_loss=0.6705 | val_acc=56.9%

Frequency-only final results:
  Accuracy: 57.3%
  AUC-ROC:  0.569
  F1:       0.383


Result: FAIL — frequency branch is below 60%.
Results saved → ./results/results.csv  (patch_exp_v4, acc=0.5731)


In [10]:
cfg_v5 = make_cfg("patch_exp_v5")
results["v5 — skin tone guided"] = run_patch_strategy(
    select_patch_v5, "v5 — skin tone guided", cfg_v5
)



Running: v5 — skin tone guided

Experiment: patch_exp_v5
Frequency-only baseline | Epochs: 20
Train: 76,848  Val: 13,561



Epoch   1/20 | train_loss=0.6869 | val_acc=55.5%
  -> Saved best val_acc=55.5%


Epoch   2/20 | train_loss=0.6795 | val_acc=56.3%
  -> Saved best val_acc=56.3%


Epoch   3/20 | train_loss=0.6744 | val_acc=55.7%


Epoch   4/20 | train_loss=0.6709 | val_acc=56.8%
  -> Saved best val_acc=56.8%


...

Epoch  16/20 | train_loss=0.6437 | val_acc=60.8%
  -> Saved best val_acc=60.8%


Epoch  17/20 | train_loss=0.6418 | val_acc=61.3%
  -> Saved best val_acc=61.3%


Epoch  18/20 | train_loss=0.6424 | val_acc=61.1%


Epoch  19/20 | train_loss=0.6411 | val_acc=60.9%


Epoch  20/20 | train_loss=0.6394 | val_acc=60.9%

Frequency-only final results:
  Accuracy: 63.2%
  AUC-ROC:  0.664
  F1:       0.457

Result: WEAK — frequency branch is below the 70% target (60-70%).
Results saved → ./results/results.csv  (patch_exp_v5, acc=0.6324)


## 7. Comparison Table

In [11]:
import pandas as pd

print("\n" + "="*55)
print("PATCH STRATEGY COMPARISON — DeepDetect")
print("Freq-only baseline, 20 epochs, test set accuracy")
print("="*55)

rows = []
for name, acc in results.items():
    if acc is None:
        continue
    status = "PASS ✓" if acc >= 0.70 else ("WEAK" if acc >= 0.60 else "FAIL ✗")
    rows.append({"Strategy": name, "Test Accuracy": f"{acc:.1%}", "Status": status})
    print(f"  {name:<38} {acc:.1%}   {status}")

print("-"*55)
print(f"  {'CIFAKE freq-only baseline (reference)':<38} 94.5%")
print("="*55)

df = pd.DataFrame(rows)
df.to_csv("patch_strategy_results.csv", index=False)
print("\nSaved: patch_strategy_results.csv")



PATCH STRATEGY COMPARISON — DeepDetect
Freq-only baseline, 20 epochs, test set accuracy
  v1 — variance only                     58.2%   FAIL ✗
  v2 — brightness filtered               62.9%   WEAK
  v4 — high-pass FFT                     57.3%   FAIL ✗
  v5 — skin tone guided                  63.2%   WEAK
-------------------------------------------------------
  CIFAKE freq-only baseline (reference)  94.5%

Saved: patch_strategy_results.csv


In [12]:
# ── Auto-select best strategy ─────────────────────────────────────────────────
strategy_fns = {
    "v1 — variance only":       select_patch_v1,
    "v2 — brightness filtered": select_patch_v2,
    "v4 — high-pass FFT":       None,
    "v5 — skin tone guided":    select_patch_v5,
}
# Pick best from results dict
best_name = max(results, key=results.get)
best_acc  = results[best_name]
BEST_STRATEGY_NAME = best_name
BEST_STRATEGY_FN   = strategy_fns[best_name]

print(f"Best strategy: {best_name}  ({best_acc:.1%})")
print(f"Auto-selected for augmentation ablation.")

# Handle v4 — high-pass has no patch fn, ablation needs special treatment
if best_name == "v4 — high-pass FFT":
    print("NOTE: v4 uses a different runner — ablation cell needs manual adjustment.")

Best strategy: v5 — skin tone guided  (63.2%)
Auto-selected for augmentation ablation.


In [12]:
from experiments.baseline_freq_only import run_freq_only_baseline as _orig_baseline
import torch.nn as nn
import torch.optim as optim
from torch.amp import GradScaler, autocast
from models.frequency_branch import FrequencyBranch
from utils.metrics import binary_accuracy
from pathlib import Path


In [16]:


def run_freq_only_with_train_acc(cfg, train_loader, val_loader, test_loader=None):
    """
    Extended freq-only baseline that also reports train accuracy per epoch.
    Fully self-contained — does not touch baseline_freq_only.py.
    """
    device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    epochs     = cfg.train.epochs
    eval_loader = test_loader if test_loader is not None else val_loader

    Path(cfg.train.checkpoint_dir).mkdir(parents=True, exist_ok=True)

    model     = FrequencyBranch(cfg.frequency, feature_dim=256).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler    = GradScaler(enabled=device.type == "cuda")

    print(f"\n{'='*70}")
    print(f"Experiment: {cfg.experiment_name}")
    print(f"Frequency-only | Epochs: {epochs}")
    print(f"{'='*70}\n")

    best_val_acc = 0.0

    for epoch in range(epochs):
        # Training
        model.train()
        total_loss   = 0.0
        train_logits = []
        train_labels = []

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            with autocast(device_type=device.type, enabled=device.type == "cuda"):
                _, aux_logits, _ = model(images)
                loss = criterion(aux_logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()
            train_logits.append(aux_logits.detach().cpu())
            train_labels.append(labels.cpu())

        scheduler.step()

        train_acc = binary_accuracy(torch.cat(train_logits), torch.cat(train_labels))

        # Val
        model.eval()
        val_logits, val_labels = [], []
        with torch.no_grad():
            for images, labels in val_loader:
                _, aux_logits, _ = model(images.to(device))
                val_logits.append(aux_logits.cpu())
                val_labels.append(labels)

        val_acc = binary_accuracy(torch.cat(val_logits), torch.cat(val_labels))

        print(f"Epoch {epoch+1:>3}/{epochs} | "
              f"train_loss={total_loss/len(train_loader):.4f} | "
              f"train_acc={train_acc:.1%} | "
              f"val_acc={val_acc:.1%}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(),
                       f"{cfg.train.checkpoint_dir}/best_{cfg.experiment_name}.pt")
            print(f"  -> Saved best val_acc={best_val_acc:.1%}")

    # Final eval on test set
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for images, labels in eval_loader:
            _, aux_logits, _ = model(images.to(device))
            all_logits.append(aux_logits.cpu())
            all_labels.append(labels)

    logits = torch.cat(all_logits)
    labels = torch.cat(all_labels)
    acc = binary_accuracy(logits, labels)
    print(f"\nTest accuracy: {acc:.1%}")
    return acc

In [ ]:
# Ablation: best strategy with and without augmentations
def make_cfg_no_aug(experiment_name):
    cfg = Config()
    cfg.data.deepdetect_root   = "../data/raw/deep_detect/data"
    cfg.data.dataset           = "deepdetect"
    cfg.data.image_size        = 224
    cfg.data.batch_size        = 64
    cfg.data.num_workers       = 4
    cfg.backbone.img_size      = 224
    cfg.frequency.patch_size   = 56
    # ── All augmentations OFF ──
    cfg.data.jpeg_aug          = False
    cfg.data.blur_aug          = False
    cfg.data.noise_aug         = False
    cfg.data.recompression_aug = False
    cfg.data.mixed_aug         = False
    cfg.train.epochs           = 20
    cfg.train.backbone_lr      = cfg.train.lr
    cfg.experiment_name        = experiment_name
    cfg.notes                  = f"patch experiment no aug: {experiment_name}"
    return cfg

# Reload data with no augmentations
cfg_no_aug = make_cfg_no_aug("patch_exp_best_no_aug")
train_loader_no_aug, val_loader_no_aug, _ = get_deepdetect_loaders(cfg_no_aug)

# Run with augmentations (reuse existing result from results dict)
acc_with_aug = results[BEST_STRATEGY_NAME]

# Run without augmentations — uses run_freq_only_with_train_acc for train acc visibility
cfg_best_no_aug = make_cfg_no_aug("patch_exp_best_no_aug")

def patched_batch_best(images, patch_size=56):
    return torch.stack([BEST_STRATEGY_FN(images[i], patch_size) for i in range(images.shape[0])])

import utils.patch_select as ps_module
import models.frequency_branch as fb_module
ps_module.select_flat_patch_batch = patched_batch_best
importlib.reload(fb_module)

try:
    acc_no_aug = run_freq_only_with_train_acc(
        cfg_best_no_aug, train_loader_no_aug, val_loader_no_aug, test_loader
    )
finally:
    from utils.patch_select import select_flat_patch_batch as _orig
    ps_module.select_flat_patch_batch = _orig
    importlib.reload(fb_module)

print("\n" + "="*55)
print(f"Augmentation ablation — {BEST_STRATEGY_NAME}")
print("="*55)
print(f"  With augmentations:     {acc_with_aug:.1%}")
print(f"  Without augmentations:  {acc_no_aug:.1%}")
print(f"  Delta:                  {acc_no_aug - acc_with_aug:+.1%}")
print("="*55)
if acc_no_aug > acc_with_aug:
    print("Augmentations are HURTING the frequency branch.")
    print("Consider disabling aug for freq-only experiments.")
else:
    print("Augmentations are HELPING or neutral.")

Train: 76,848  Val: 13,561  Test: 21,776

Experiment: patch_exp_best_no_aug
Frequency-only | Epochs: 20

Epoch   1/20 | train_loss=0.6653 | train_acc=59.0% | val_acc=59.3%
  -> Saved best val_acc=59.3%
Epoch   2/20 | train_loss=0.6244 | train_acc=64.5% | val_acc=61.6%
  -> Saved best val_acc=61.6%
Epoch   3/20 | train_loss=0.6055 | train_acc=66.0% | val_acc=63.9%
  -> Saved best val_acc=63.9%
Epoch   4/20 | train_loss=0.5922 | train_acc=67.6% | val_acc=66.4%
  -> Saved best val_acc=66.4%
Epoch   5/20 | train_loss=0.5852 | train_acc=68.3% | val_acc=62.8%
Epoch   6/20 | train_loss=0.5750 | train_acc=69.1% | val_acc=67.5%
  -> Saved best val_acc=67.5%
Epoch   7/20 | train_loss=0.5674 | train_acc=69.6% | val_acc=66.8%
Epoch   8/20 | train_loss=0.5612 | train_acc=70.1% | val_acc=68.7%
  -> Saved best val_acc=68.7%
Epoch   9/20 | train_loss=0.5552 | train_acc=70.5% | val_acc=63.5%
Epoch  10/20 | train_loss=0.5505 | train_acc=71.0% | val_acc=64.4%
Epoch  11/20 | train_loss=0.5432 | train_acc=

In [ ]:
# Follow-up: v5 no augmentations, 30 epochs
# realistic augmentations calibrated to real-world degradation levels.

def make_cfg_v5_no_aug(experiment_name):
    cfg = Config()
    cfg.data.deepdetect_root   = "../data/raw/deep_detect/data"
    cfg.data.dataset           = "deepdetect"
    cfg.data.image_size        = 224
    cfg.data.batch_size        = 64
    cfg.data.num_workers       = 4
    cfg.backbone.img_size      = 224
    cfg.frequency.patch_size   = 56
    cfg.data.jpeg_aug          = False
    cfg.data.blur_aug          = False
    cfg.data.noise_aug         = False
    cfg.data.recompression_aug = False
    cfg.data.mixed_aug         = False
    cfg.train.epochs           = 30
    cfg.train.backbone_lr      = cfg.train.lr
    cfg.experiment_name        = experiment_name
    cfg.notes                  = "v5 skin tone, no aug, 30 epochs — frequency ceiling"
    return cfg

cfg_v5_ceiling = make_cfg_v5_no_aug("patch_exp_v5_no_aug_30ep")
train_loader_clean, val_loader_clean, _ = get_deepdetect_loaders(cfg_v5_ceiling)

def patched_batch_v5(images, patch_size=56):
    return torch.stack([select_patch_v5(images[i], patch_size)
                        for i in range(images.shape[0])])

ps_module.select_flat_patch_batch = patched_batch_v5
importlib.reload(fb_module)

try:
    acc_v5_ceiling = run_freq_only_with_train_acc(
        cfg_v5_ceiling, train_loader_clean, val_loader_clean, test_loader
    )
finally:
    from utils.patch_select import select_flat_patch_batch as _orig
    ps_module.select_flat_patch_batch = _orig
    importlib.reload(fb_module)

print("\n" + "="*55)
print("v5 frequency branch ceiling (no augmentations)")
print("="*55)
print(f"  20 epochs with aug:     63.2%")
print(f"  20 epochs no aug:       72.5%")
print(f"  30 epochs no aug:       {acc_v5_ceiling:.1%}")
print("="*55)

Train: 76,848  Val: 13,561  Test: 21,776

Experiment: patch_exp_v5_no_aug_30ep
Frequency-only | Epochs: 30

Epoch   1/30 | train_loss=0.6706 | train_acc=58.4% | val_acc=57.3%
  -> Saved best val_acc=57.3%
Epoch   2/30 | train_loss=0.6319 | train_acc=63.6% | val_acc=61.4%
  -> Saved best val_acc=61.4%
Epoch   3/30 | train_loss=0.6128 | train_acc=65.4% | val_acc=62.5%
  -> Saved best val_acc=62.5%
Epoch   4/30 | train_loss=0.6004 | train_acc=66.8% | val_acc=62.6%
  -> Saved best val_acc=62.6%
Epoch   5/30 | train_loss=0.5902 | train_acc=67.6% | val_acc=64.5%
  -> Saved best val_acc=64.5%
Epoch   6/30 | train_loss=0.5801 | train_acc=68.6% | val_acc=64.8%
  -> Saved best val_acc=64.8%
Epoch   7/30 | train_loss=0.5750 | train_acc=69.0% | val_acc=64.7%
Epoch   8/30 | train_loss=0.5689 | train_acc=69.4% | val_acc=66.4%
  -> Saved best val_acc=66.4%
Epoch   9/30 | train_loss=0.5616 | train_acc=70.1% | val_acc=65.6%
Epoch  10/30 | train_loss=0.5592 | train_acc=70.4% | val_acc=68.1%
  -> Saved b